# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
spaceship.shape


(8693, 14)

**Check for data types**

In [4]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [5]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
spaceship = spaceship.dropna().reset_index(drop=True)
spaceship.shape


(6606, 14)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [7]:
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]
spaceship["Cabin"].unique()[:10]

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

- Drop PassengerId and Name

In [8]:
spaceship = spaceship.drop(columns=["PassengerId","Name"], errors="ignore")


- For non-numerical columns, do dummies.

In [9]:
# Identify non numerical columns 
non_numeric = spaceship.select_dtypes(include="object").columns

spaceship_encoded = pd.get_dummies(spaceship, columns=non_numeric, drop_first=True)
spaceship_encoded.head(2)


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False


##  Alternative Full Pipeline 

Below we apply a more advanced version of the preprocessing and modeling steps using a **ColumnTransformer** and a **Pipeline** from `scikit-learn`.  
This version automatically handles both:
- Scaling for numerical features  
- One-Hot Encoding for categorical features  

**Perform Train Test Split**

In [10]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# 1️ Define X and y
X = spaceship.drop(columns=["Transported", "PassengerId", "Name"], errors="ignore")
y = spaceship["Transported"].astype(int)

# 2️ Identify categorical and numeric columns
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(include=["float64", "int64"]).columns

# 3️ Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ])

# 4️ Full pipeline with model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn', KNeighborsClassifier(n_neighbors=7))
])

# 5️ Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 6️ Fit and evaluate
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7723

Confusion Matrix:
 [[502 154]
 [147 519]]

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.77      0.77       656
           1       0.77      0.78      0.78       666

    accuracy                           0.77      1322
   macro avg       0.77      0.77      0.77      1322
weighted avg       0.77      0.77      0.77      1322



In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X y y
X = spaceship_encoded.drop(columns="Transported")
y = spaceship_encoded["Transported"].astype(int)  # 0/1

# split the dataset; afterwards, we applied scaling using only the training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# We scaled all the columns in X (they are already numeric)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#Model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

#Prediction
y_pred = knn.predict(X_test_scaled)

#Evaluation 
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred, digits=3))


Accuracy: 0.7678

Matriz de confusión:
 [[489 167]
 [140 526]]

Reporte de clasificación:
               precision    recall  f1-score   support

           0      0.777     0.745     0.761       656
           1      0.759     0.790     0.774       666

    accuracy                          0.768      1322
   macro avg      0.768     0.768     0.768      1322
weighted avg      0.768     0.768     0.768      1322



- Evaluate your model's performance. Comment it

In [13]:
for k in [3,5,7,9,11,15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_scaled))
    print(f"k={k} → accuracy={acc:.4f}")

k=3 → accuracy=0.7511
k=5 → accuracy=0.7678
k=7 → accuracy=0.7708
k=9 → accuracy=0.7799
k=11 → accuracy=0.7769
k=15 → accuracy=0.7799


The KNN model achieved an accuracy of around 0.78.
This means the model correctly predicts about 78% of the passengers’ transport status.
Considering that we used a simple model and minimal feature engineering, the performance is acceptable.
Feature scaling was crucial because KNN relies on distances, and scaling ensures all features contribute equally.